# Train YOLO With Configurable Data Path

Notebook nay gom 3 phan:

1. Cấu hình đường dẫn dataset va tham so train.
2. Kiem tra nhanh dataset YOLO.
3. Train baseline YOLO va scaffold cho XAI-guided training.


In [ ]:
!git clone https://github.com/Thanhmay2406/xai-driven-saliency-loss.git

In [ ]:
from pathlib import Path

CFG = {
    "DATASET_YAML": "data/merged_yolo_grouped/dataset.yaml",
    "MODEL_WEIGHTS": "yolo11n.pt",
    "PROJECT_DIR": "outputs/yolo",
    "BASELINE_RUN_NAME": "baseline",
    "XAI_RUN_NAME": "xai_notebook",
    "EPOCHS": 100,
    "BATCH": 16,
    "IMGSZ": 640,
    "DEVICE": 0,
    "WORKERS": 4,
    "PATIENCE": 20,
    "SEED": 42,
    "LR": 1e-3,
    "WEIGHT_DECAY": 5e-4,
    "XAI_METHOD": "gradcam",
    "LAMBDA_SALIENCY": 0.1,
    "NUM_CLASSES": 6,
    "USE_PROGRESSIVE_LAMBDA": True,
    "PROGRESSIVE_WARMUP_EPOCHS": 20,
    "RUN_BASELINE": True,
    "SHOW_XAI_TEMPLATE": True,
}

REPO_ROOT = Path.cwd()
DATASET_YAML = (REPO_ROOT / CFG["DATASET_YAML"]).resolve()
print("Dataset YAML:", DATASET_YAML)


In [ ]:
import yaml

if not DATASET_YAML.exists():
    raise FileNotFoundError(f"Khong tim thay dataset yaml: {DATASET_YAML}")

with DATASET_YAML.open("r", encoding="utf-8") as f:
    dataset_cfg = yaml.safe_load(f) or {}

dataset_root = Path(dataset_cfg.get("path", DATASET_YAML.parent))
if not dataset_root.is_absolute():
    dataset_root = (DATASET_YAML.parent / dataset_root).resolve()

print("Dataset root:", dataset_root)
for split_name in ("train", "val", "test"):
    split_rel = dataset_cfg.get(split_name)
    if split_rel:
        split_path = (dataset_root / split_rel).resolve()
        print(f"{split_name}: {split_path} | exists={split_path.exists()}")

print("Classes:", dataset_cfg.get("names"))


## Optional Install

Neu kernel cua ban chua co `ultralytics`, mo comment cell duoi day va chay mot lan.


In [ ]:
# !pip install ultralytics pyyaml


## Baseline YOLO Training

Cell nay dung flow baseline YOLO thong thuong va da duoc canh lai theo `train_baseline.ipynb`. Duong dan dataset duoc lay tu `CFG["DATASET_YAML"]`.


In [ ]:
if CFG["RUN_BASELINE"]:
    from ultralytics import YOLO

    baseline_model = YOLO(CFG["MODEL_WEIGHTS"])
    baseline_results = baseline_model.train(
        data=str(DATASET_YAML),
        project=CFG["PROJECT_DIR"],
        name=CFG["BASELINE_RUN_NAME"],
        epochs=CFG["EPOCHS"],
        imgsz=CFG["IMGSZ"],
        batch=CFG["BATCH"],
        device=CFG["DEVICE"],
        workers=CFG["WORKERS"],
        patience=CFG["PATIENCE"],
        seed=CFG["SEED"],
    )
    baseline_results
else:
    print("RUN_BASELINE=False, bo qua baseline training.")


## XAI Training Template

Cell duoi day la scaffold de noi voi `UltralyticsYOLOXAITrainer` da tao trong repo. No phu hop khi ban co model va dataloader tra ve batch dict theo format Ultralytics: `img`, `bboxes`, `cls`, `batch_idx`.


In [ ]:
if CFG["SHOW_XAI_TEMPLATE"]:
    from src.training import UltralyticsYOLOXAITrainer, UltralyticsYOLOXAITrainerConfig

    print("XAI trainer template is available.")
    print("Use the config below after you wire a YOLO-style dataloader.")

    xai_cfg = UltralyticsYOLOXAITrainerConfig(
        xai_method=CFG["XAI_METHOD"],
        lambda_saliency=CFG["LAMBDA_SALIENCY"],
        num_classes=CFG["NUM_CLASSES"],
        use_progressive_lambda=CFG["USE_PROGRESSIVE_LAMBDA"],
        progressive_warmup_epochs=CFG["PROGRESSIVE_WARMUP_EPOCHS"],
    )
    xai_cfg
else:
    print("SHOW_XAI_TEMPLATE=False")


In [ ]:
# Template only. Uncomment and adapt after you have a YOLO-style train_loader.
#
# from ultralytics import YOLO
# import torch
#
# yolo = YOLO(CFG["MODEL_WEIGHTS"])
# model = yolo.model
# optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
# trainer = UltralyticsYOLOXAITrainer(model=model, optimizer=optimizer, config=xai_cfg)
#
# for epoch in range(CFG["EPOCHS"]):
#     for batch in train_loader:
#         step_output = trainer.training_step(batch, epoch=epoch)
#         print(
#             step_output.detection_loss.item(),
#             step_output.saliency_loss.item(),
#             step_output.total_loss.item(),
#         )
#
# trainer.close()
